# NVIDIA Nemotron-Mini-4B on Snowpark Container Services

**What this notebook shows:**
1. Verify the NIM inference service is running
2. Call Nemotron-Mini-4B from Python using the OpenAI SDK (REST path)
3. Call Nemotron-Mini-4B from SQL using `NIM_COMPLETE()` (no API key needed)
4. Run batch inference directly on Snowflake data

**Architecture:**

![Architecture diagram](../../img/architecture.png)

> Source: [`../img/architecture.excalidraw`](../img/architecture.excalidraw) — open in [Excalidraw](https://excalidraw.com) to edit.

**Prerequisites:** `setup/01_snowflake_setup.sql` and `setup/02_deploy_service.sql` must be executed first.


## Setup

In [1]:
import pandas as pd

try:
    # Running inside Snowflake Notebooks (Snowsight)
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Running locally — uses ~/.snowflake/connections.toml or config
    from snowflake.snowpark import Session
    # Change CONNECTION_NAME to match your entry in connections.toml
    # e.g. 'MYORG_AWS_US_EAST_1'
    CONNECTION_NAME = "YOUR_CONNECTION_NAME"  # replace with your connection name from ~/.snowflake/connections.toml
    session = Session.builder.config("connection_name", CONNECTION_NAME).create()

session.sql("USE SCHEMA NEMOTRON_DB.NEMOTRON_SCHEMA").collect()

print("Connected as:", session.get_current_user())
print("Role:", session.get_current_role())
print("Database:", session.get_current_database())


Connected as: "JMARCISZEWSKI"
Role: "ACCOUNTADMIN"
Database: "NEMOTRON_DB"


## 1. Verify Service Status

The NIM container takes **5–15 minutes on its first cold start** while it:
1. Downloads `nvidia/Nemotron-Mini-4B-Instruct` weights from HuggingFace (~8 GB)
2. Compiles a TensorRT-LLM engine optimized for the A10G GPU

Subsequent restarts are much faster because the compiled engine is cached on the block volume.


In [2]:
import json

status_raw = session.sql(
    "SELECT SYSTEM$GET_SERVICE_STATUS('NEMOTRON_DB.NEMOTRON_SCHEMA.NEMOTRON_SERVICE')"
).collect()[0][0]

status = json.loads(status_raw)
for container in status:
    name   = container.get("containerName", "?")
    state  = container.get("status", "?")
    msg    = container.get("message", "")
    # SPCS omits the "ready" field when status=READY; infer readiness from status
    ready  = (state == "READY")
    icon   = "\u2705" if ready else "\u23f3"
    print(f"{icon} [{name}]  status={state}  ready={ready}  {msg}")


✅ [nim]  status=READY  ready=True  Running
✅ [translator]  status=READY  ready=True  Running. WARN: missing 'nvidia.com/gpu' as part of the container specification. For more information please see https://docs.snowflake.com/en/developer-guide/snowpark-container-services/specification-reference#containers-resources-field.


In [3]:
# Get the public endpoint URL — use this for direct REST access
endpoints = session.sql(
    "SHOW ENDPOINTS IN SERVICE NEMOTRON_DB.NEMOTRON_SCHEMA.NEMOTRON_SERVICE"
).to_pandas()
# SHOW commands return column names wrapped in double-quote chars — strip them
endpoints.columns = [c.strip('"') for c in endpoints.columns]
print(endpoints[["name", "port", "ingress_url"]].to_string(index=False))

# Save the NIM API public URL for later cells
nim_row = endpoints[endpoints["name"] == "nim-api"]
NIM_INGRESS_URL = nim_row["ingress_url"].values[0] if len(nim_row) else None
print(f"\nNIM public endpoint: https://{NIM_INGRESS_URL}")


        name  port                                                       ingress_url
     nim-api  5000 i3d7p3-sfsenorthamerica-jmarciszewski-aws1.snowflakecomputing.app
nim-internal  8000                                                              None

NIM public endpoint: https://i3d7p3-sfsenorthamerica-jmarciszewski-aws1.snowflakecomputing.app


## 2. Python Inference via OpenAI SDK

The NIM service is OpenAI-compatible.  Authentication uses your Snowflake session token
(no separate API key needed for inference — RBAC is handled by Snowflake).


In [4]:
import os
from openai import OpenAI

# Get auth token for the SPCS public endpoint.
# PAT-based connections (PROGRAMMATIC_ACCESS_TOKEN): use the PAT directly via
# conn._token_file_path — derived session tokens do not pass SPCS endpoint auth.
# Snowsight notebooks / username+password / keypair connections: fall back to
# _token_request("ISSUE") which returns a short-lived session token.
conn = session._conn._conn
_token_file = getattr(conn, "_token_file_path", None)
if _token_file and os.path.isfile(_token_file):
    sf_token = open(_token_file).read().strip()   # PAT connection
else:
    sf_token = conn._rest._token_request("ISSUE")["data"]["sessionToken"]

client = OpenAI(
    base_url=f"https://{NIM_INGRESS_URL}/v1",
    api_key="not-used",
    default_headers={"Authorization": f'Snowflake Token="{sf_token}"'},
)

# Auto-discover the model name from NIM via the translator proxy
NIM_MODEL = client.models.list().data[0].id
print(f"Model: {NIM_MODEL}")

response = client.chat.completions.create(
    model=NIM_MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a concise, technically accurate AI assistant."
        },
        {
            "role": "user",
            "content": "Explain in 3 bullet points why enterprises run LLM inference inside Snowflake instead of calling external APIs."
        }
    ],
    max_tokens=512,
)

print(response.choices[0].message.content)
print(f"\n--- Usage: {response.usage} ---")


Model: hf://nvidia/Nemotron-Mini-4B-Instruct


* Advantage of Snowflake for LLM inference: Snowflake's serverless architecture and petabyte-scale data warehouse enable fast, scalable, and cost-effective LLM inference. Snowflake's auto-scaling and near-linear scaling capabilities ensure consistent performance and efficient resource utilization, reducing the need for dedicated server infrastructure and associated maintenance costs. Additionally, Snowflake's on-premises or hybrid cloud capabilities offer control over security, compliance, and infrastructure management, which are crucial for enterprise-grade applications.

* Benefits of internal LLM inference: By running LLM inference locally within Snowflake, enterprises enjoy faster latency, lower latency variability (since Snowflake is edge-optimized and caching enabled), and better control over data privacy and security. Internal inference also eliminates the need for external API calls, data transmission, and potential latency introduced by data replication and API throttling. Thi

## 3. SQL Inference with `NIM_COMPLETE()`

The `NIM_COMPLETE` service function lets any SQL query call Nemotron — no Python, no REST,
no API key management.  It's a first-class Snowflake scalar function.


In [5]:
prompts = [
    "What is Snowpark Container Services?",
    "Summarize NVIDIA NIM in one sentence.",
    "Why is data governance important for enterprise AI?",
    "What does TensorRT-LLM optimize?",
]

for p in prompts:
    result = session.sql(
        "SELECT NEMOTRON_DB.NEMOTRON_SCHEMA.NIM_COMPLETE(?)", params=[p]
    ).collect()[0][0]
    print(f"Q: {p}")
    print(f"A: {result}\n{'─'*60}")


Q: What is Snowpark Container Services?
A:  Snowpark Container Services is a company that specializes in the design, manufacture, and installation of premium floating container living units, commonly known as snowparkers or icetents. These unique structures are built specifically for use during winter activities such as skiing, snowboarding, and ice fishing in snowy environments.

Here are some key features and benefits of the Snowpark Container Services products:

1. Modular and customizable: Snowpark containers come in various sizes, from basic 4x8 foot models to luxurious 25x80 foot units, allowing customers to choose the size and style that best suits their needs. They can also be easily customized with walls, flooring, insulation, and other features to create a cozy and comfortable living space.

2. High-quality materials: Snowpark containers are made of high-quality materials, such as steel, aluminum, and IDC-coated polycarbonate (a soft, impact-resistant material). These materia

Q: Summarize NVIDIA NIM in one sentence.
A:  NVIDIA NIM (Network Inference Modules) is a suite of pre-trained AI models and inference optimizations designed to streamline and accelerate the deployment of deep learning models for various applications.
────────────────────────────────────────────────────────────


Q: Why is data governance important for enterprise AI?
A:  Data governance is crucial for enterprise AI for several reasons, ensuring that data is collected, organized, analyzed, and utilized ethically, effectively, and efficiently. Here are some key points to consider:

1. **Data Quality and Integrity**: Effective data governance practices help maintain data quality and integrity. This includes implementing data validation rules, ensuring data is accurate and up-to-date, and handling missing or inconsistent data. High-quality data leads to more accurate AI models and better business outcomes.

2. **Data Privacy and Security**: Data governance policies and procedures ensure that data is handled, stored, and shared in compliance with data protection regulations. This helps maintain customer trust, prevents data breaches, and reduces legal risks associated with AI model development and deployment.

3. **Data Security**: Robust data governance frameworks protect sensitive data from unauth

Q: What does TensorRT-LLM optimize?
A:  TensorRT-LLM, also known as TensorRT for Large Language Models, is a deep learning optimization framework developed by NVIDIA for accelerating the deployment of neural network models, particularly those used in large language models (LLMs). Here's a breakdown of what TensorRT-LLM optimizes:

1. Model quantization: TensorRT-LLM optimizes the quantization process, which is the process of reducing the precision of the model's weights and activations from 32-bit floating-point to lower-precision formats like 16-bit or 8-bit. This optimization reduces memory footprint, improves throughput, and enables faster inference. TensorRT supports various quantization algorithms and levels of precision, from simple quantization to more advanced techniques like dynamic or mixed-precision quantization.

2. Mixed-precision training: TensorRT-LLM also supports mixed-precision training, which allows for precision control during training, balancing the benefits of low

## 4. Batch Inference on Snowflake Data

`NIM_COMPLETE` works natively in any SELECT — run inference directly on rows in a table.
This demo creates a small sample table and enriches it with LLM-generated summaries.


In [6]:
# Create a small demo table of product descriptions
session.sql("""
    CREATE OR REPLACE TEMP TABLE DEMO_PRODUCTS (
        product_id   INT,
        product_name VARCHAR,
        raw_description VARCHAR
    )
""").collect()

session.sql("""
    INSERT INTO DEMO_PRODUCTS VALUES
      (1, 'Snowflake Data Cloud',    'A cloud-based data warehousing platform that enables storage, processing, and analytics.'),
      (2, 'NVIDIA NIM',              'Microservices for deploying AI models with TensorRT-LLM optimization and OpenAI-compatible APIs.'),
      (3, 'Snowpark Container Svcs', 'A runtime for deploying containerized workloads including ML inference directly within Snowflake.'),
      (4, 'Nemotron-Mini-4B',        'A 4B-parameter LLM from NVIDIA fine-tuned for instruction following and chat applications.')
""").collect()

print("Sample data:")
session.sql("SELECT * FROM DEMO_PRODUCTS").show()


Sample data:


-----------------------------------------------------------------------------------------------
|"PRODUCT_ID"  |"PRODUCT_NAME"           |"RAW_DESCRIPTION"                                   |
-----------------------------------------------------------------------------------------------
|1             |Snowflake Data Cloud     |A cloud-based data warehousing platform that en...  |
|2             |NVIDIA NIM               |Microservices for deploying AI models with Tens...  |
|3             |Snowpark Container Svcs  |A runtime for deploying containerized workloads...  |
|4             |Nemotron-Mini-4B         |A 4B-parameter LLM from NVIDIA fine-tuned for i...  |
-----------------------------------------------------------------------------------------------



In [7]:
# Enrich each row with an LLM-generated customer-friendly tagline
results = session.sql("""
    SELECT
        product_id,
        product_name,
        NEMOTRON_DB.NEMOTRON_SCHEMA.NIM_COMPLETE(
            'Write a punchy one-sentence marketing tagline for: ' || raw_description
        ) AS ai_tagline
    FROM DEMO_PRODUCTS
    ORDER BY product_id
""").to_pandas()

for _, row in results.iterrows():
    print(f"[{row['PRODUCT_ID']}] {row['PRODUCT_NAME']}")
    print(f"    → {row['AI_TAGLINE']}\n")


[1] Snowflake Data Cloud
    →  "Unlock insights with our cloud-based data warehousing, empowering your business with scalable, secure, and actionable data processing and analytics."

[2] NVIDIA NIM
    →  "Empower AI innovation with microservices, guaranteed performance with TensorRT-LLM, and seamless integration with OpenAI-compatible APIs."

[3] Snowpark Container Svcs
    →  "Accelerate your ML-powered Snowflake applications with seamless containerization and on-demand inference, streamlining data governance and computational efficiency."

[4] Nemotron-Mini-4B
    →  "NVIDIA's 4B LLM, fine-tuned for seamless instruction following and engaging conversation, powering innovative AI applications."



## 5. Cost & Operations Tips

In [8]:
tips = """
GPU Cost Management
───────────────────
• GPU_NV_S bills by the second while the compute pool is ACTIVE.
• The AUTO_SUSPEND_SECS = 3600 setting in 01_snowflake_setup.sql suspends
  the pool after 1 hour of inactivity.
• Manually suspend/resume when you control the schedule:

    ALTER COMPUTE POOL NEMOTRON_GPU_POOL SUSPEND;
    ALTER COMPUTE POOL NEMOTRON_GPU_POOL RESUME;

Block Storage
─────────────
• The 50 GB block volume caches the compiled TensorRT-LLM engine.
• First cold start: 5–15 min (weight download + TRT compilation).
• Warm restart (cache hit): < 2 min.
• Without block storage, every restart re-downloads ~8 GB.

Scaling
───────
• This demo uses MIN_INSTANCES = MAX_INSTANCES = 1 (single node).
• For production throughput, increase MAX_INSTANCES — NIM auto-scales.

LoRA Adapters (advanced)
────────────────────────
• NIM supports hot-swapping LoRA fine-tuned adapters without rebuilding containers.
• Fine-tune Nemotron on your domain data, register the adapter, and NIM loads it at request time.
"""
print(tips)



GPU Cost Management
───────────────────
• GPU_NV_S bills by the second while the compute pool is ACTIVE.
• The AUTO_SUSPEND_SECS = 3600 setting in 01_snowflake_setup.sql suspends
  the pool after 1 hour of inactivity.
• Manually suspend/resume when you control the schedule:

    ALTER COMPUTE POOL NEMOTRON_GPU_POOL SUSPEND;
    ALTER COMPUTE POOL NEMOTRON_GPU_POOL RESUME;

Block Storage
─────────────
• The 50 GB block volume caches the compiled TensorRT-LLM engine.
• First cold start: 5–15 min (weight download + TRT compilation).
• Warm restart (cache hit): < 2 min.
• Without block storage, every restart re-downloads ~8 GB.

Scaling
───────
• This demo uses MIN_INSTANCES = MAX_INSTANCES = 1 (single node).
• For production throughput, increase MAX_INSTANCES — NIM auto-scales.

LoRA Adapters (advanced)
────────────────────────
• NIM supports hot-swapping LoRA fine-tuned adapters without rebuilding containers.
• Fine-tune Nemotron on your domain data, register the adapter, and NIM loads 